# Gold-set baseline run + scoring
Runtime → Change runtime type → **T4 GPU**. Run the cells **in order**.

In [1]:
# 1. Clone (or update) the repo and install dependencies
REPO_URL = "https://github.com/m-mirkan/peft-cpe-extraction-pipeline.git"
BRANCH = "feat/codebook-data-curation"

import os
if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Cloning into '/content/repo'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 77 (delta 26), reused 48 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 2.52 MiB | 5.64 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/repo
Already on 'feat/codebook-data-curation'
Your branch is up to date with 'origin/feat/codebook-data-curation'.
From https://github.com/m-mirkan/peft-cpe-extraction-pipeline
 * branch            feat/codebook-data-curation -> FETCH_HEAD
Already up to date.
CUDA available: True | GPU: Tesla T4


In [2]:
# 2. Download baseline adapter
import os, zipfile
LORA_GDRIVE_ID = "1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX"
LORA_DIR = "checkpoints/qwen25_3b_cpe_lora"
os.makedirs(LORA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(LORA_DIR, "adapter_config.json")):
    !gdown {LORA_GDRIVE_ID} -O lora_adapter.zip
    with zipfile.ZipFile("lora_adapter.zip", "r") as z:
        z.extractall(LORA_DIR)
    entries = os.listdir(LORA_DIR)
    if len(entries) == 1 and os.path.isdir(os.path.join(LORA_DIR, entries[0])):
        nested = os.path.join(LORA_DIR, entries[0])
        for f in os.listdir(nested):
            os.rename(os.path.join(nested, f), os.path.join(LORA_DIR, f))
        os.rmdir(nested)
print("LoRA adapter files:", os.listdir(LORA_DIR))

Downloading...
From (original): https://drive.google.com/uc?id=1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX
From (redirected): https://drive.google.com/uc?id=1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX&confirm=t&uuid=a8cd580a-9b6e-41b4-938b-552fa5f35836
To: /content/repo/lora_adapter.zip
100% 113M/113M [00:02<00:00, 37.6MB/s]
LoRA adapter files: ['tokenizer.json', 'chat_template.jinja', 'README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json']


In [3]:
# 3. Sanity check: the gold files and the patched script must be there
import json
!ls data/gold scripts
inputs = json.load(open("data/gold/gold_inputs.json", encoding="utf-8"))
gold = json.load(open("data/gold/gold_template.json", encoding="utf-8"))
print("gold titles:", len(inputs), "| unique:", len(set(inputs)), "| labeled:", sum(g["status"] == "labeled" for g in gold))
assert len(inputs) == 100 and [g["input"] for g in gold] == inputs
print("patched inference script (should print 2 lines):")
!grep -n "max_new_tokens=2048\|FAILED" src/infer_and_resolve.py

data/gold:
gold_inputs.json  gold_review.md  gold_template.json

scripts:
check_leakage.py  score_gold.py
gold titles: 100 | unique: 100 | labeled: 100
patched inference script (should print 2 lines):
86:            raw_json = model(prompt, TOEExtraction, max_new_tokens=2048)
93:            print(f"\n[{i}] FAILED: {raw_text}\n  {err}")


In [4]:
# 4. Run titles 1-50 (saves after every title; failures are recorded, not fatal)
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 0 --limit 50 --output_file outputs/gold_base_p1.json

[Infer] Loading base model `Qwen/Qwen2.5-3B-Instruct` on cuda (torch.bfloat16)...
config.json: 100% 661/661 [00:00<00:00, 2.92MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 77.0MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.97G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   1% 67.0M/6.17G [00:01<01:38, 62.1MB/s, 30.0kB/s  ]
Reconstructing (incomplete total...):   4% 226M/6.17G [00:01<00:26, 225MB/s,  146MB/s  ]
Reconstructing (incomplete total...):   9% 560M/6.17G [00:06<01:30, 61.8MB/s,  159MB/s  ]
Reconstructing (incomplete total...):  50% 3.08G/6.17G [00:31<00:31, 98.1MB/s, 59.7MB/s  ]
Reconstructing (incomplete total...):  79% 4.88G/6.17G [00:45<00:09, 135MB/s,  151MB/s  ]

Fetchin

In [5]:
# 5. Run titles 51-100
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 50 --limit 50 --output_file outputs/gold_base_p2.json

[Infer] Loading base model `Qwen/Qwen2.5-3B-Instruct` on cuda (torch.bfloat16)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:12<00:00, 35.75it/s]
[Infer] Attaching fine-tuned LoRA adapter from `checkpoints/qwen25_3b_cpe_lora`...

[51] INPUT: MIFARE DESFire EV3 on ST31R480 A01(version 1.0.3) (ANSSI-CC-2025/31)
---------------------------------------------------------------------------
  h:stmicroelectronics:st31r480_a01:-                [explicit_vendor]
  h:nxp:mifare_desfire_ev3:-                         [explicit_vendor]
  a:stmicroelectronics:anssi-cc-2025_31:-            [clause_inheritance]

[52] INPUT: NXP JCOP 5.2 on SN100.C58 Secure Element
---------------------------------------------------------------------------
  a:nxp:jcop:5.2                                     [explicit_vendor]
  h:nxp:sn100.c58:-                                  [explicit_vendor]

[53] INPUT: EMC® Ionix™ for IT Operations Intelligence (SMARTS®) - S

In [6]:
# 6. Score against the gold set
import json
n1 = len(json.load(open("outputs/gold_base_p1.json", encoding="utf-8")))
n2 = len(json.load(open("outputs/gold_base_p2.json", encoding="utf-8")))
print("predictions saved:", n1, "+", n2, "(expected 50 + 50)")
!python scripts/score_gold.py --gold data/gold/gold_template.json --pred baseline=outputs/gold_base_p1.json,outputs/gold_base_p2.json --json_out outputs/gold_base_scores.json

predictions saved: 50 + 50 (expected 50 + 50)
Scored titles: 100 (status=labeled). Each title is worth 1.0 points of title-exact-match.

== ALL ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
baseline          100    0   0.56   0.54   0.55     0.95       0.29      0.60

== difficulty:easy ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
baseline           30    0   0.55   0.55   0.55     0.94       0.53      0.61

== difficulty:hard ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
baseline           29    0   0.63   0.58   0.60     0.94       0.07      0.65

== difficulty:medium ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
baseline           41    0   0.44   0.45   0.44     0.96       0.27      0.52

== enterprise_sw ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
baseline           18    0   0.50   0.52   0.51     0.94   

In [7]:
# 7. Download the outputs
from google.colab import files
files.download("outputs/gold_base_p1.json")
files.download("outputs/gold_base_p2.json")
files.download("outputs/gold_base_scores.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>